# Decision Tree

Notebook này dùng để note lại kiến thức về **Decision Tree** kèm công thức, ví dụ tính tay và demo bằng Python.

Nội dung chính:

1. Decision Tree là gì?
2. Cách cây quyết định chia nhánh
3. Công thức Gini Impurity
4. Công thức Entropy
5. Information Gain
6. Ví dụ tính tay
7. Demo Python với `sklearn`
8. Ưu điểm và nhược điểm
9. Bài tập luyện tập

## 1. Decision Tree là gì?

**Decision Tree** là thuật toán Machine Learning dùng cấu trúc dạng cây để đưa ra quyết định.

Mỗi cây gồm các thành phần chính:

| Thành phần | Ý nghĩa |
|---|---|
| Root Node | Node đầu tiên của cây |
| Internal Node | Node ở giữa, dùng để đặt câu hỏi/chia dữ liệu |
| Branch | Nhánh đi ra từ một điều kiện |
| Leaf Node | Node cuối cùng, chứa kết quả dự đoán |

Ví dụ:

```text
Hours Study <= 4?
├── Yes → Fail
└── No  → Pass
```

Ý tưởng chính:

> Decision Tree liên tục đặt câu hỏi để chia dữ liệu thành các nhóm nhỏ hơn, sao cho mỗi nhóm càng “sạch” càng tốt.

## 2. Cây quyết định chia nhánh như thế nào?

Decision Tree sẽ thử nhiều cách chia dữ liệu khác nhau.

Ví dụ:

```text
Chia theo Hours Study <= 4?
Chia theo Hours Study <= 5?
Chia theo Attendance <= 70?
Chia theo Income <= 500?
```

Sau đó cây chọn cách chia tốt nhất.

Một cách chia tốt là cách chia làm cho dữ liệu sau khi tách ra trở nên **pure** hơn.

### Pure nghĩa là gì?

Một node được gọi là **pure** nếu trong node đó hầu như chỉ có một class.

Ví dụ node sạch:

```text
5 Pass, 0 Fail → rất pure
```

Node bị lẫn:

```text
3 Pass, 3 Fail → không pure
```

Nói đơn giản:

```text
Decision Tree chọn feature và threshold sao cho sau khi chia, mỗi nhánh càng ít lẫn class càng tốt.
```

## 3. Gini Impurity

**Gini Impurity** đo mức độ lẫn lộn của một node.

Công thức:

$$
Gini = 1 - \sum p_i^2
$$

Trong đó:

$$
p_i = \text{tỷ lệ của class } i \text{ trong node}
$$

Với bài toán 2 class:

$$
Gini = 1 - (p_1^2 + p_2^2)
$$

Ý nghĩa:

| Gini | Ý nghĩa |
|---:|---|
| 0 | Node rất sạch, chỉ có một class |
| Gần 0.5 | Node bị lẫn nhiều class |

Ví dụ node có 8 Positive và 2 Negative:

$$
p_{pos} = \frac{8}{10} = 0.8
$$

$$
p_{neg} = \frac{2}{10} = 0.2
$$

$$
Gini = 1 - (0.8^2 + 0.2^2)
$$

$$
Gini = 1 - (0.64 + 0.04) = 0.32
$$

Kết luận:

```text
Gini = 0.32 → node tương đối sạch vì Positive chiếm đa số.
```

In [ ]:
# Tính Gini Impurity bằng Python

def gini(counts):
    total = sum(counts)
    probabilities = [count / total for count in counts]
    return 1 - sum(p ** 2 for p in probabilities)

# Ví dụ: 8 Positive, 2 Negative
gini_value = gini([8, 2])
gini_value

## 4. Entropy

**Entropy** cũng đo mức độ hỗn loạn/lẫn lộn của một node.

Công thức:

$$
Entropy = - \sum p_i \log_2(p_i)
$$

Trong đó:

$$
p_i = \text{tỷ lệ của class } i \text{ trong node}
$$

Ý nghĩa:

| Entropy | Ý nghĩa |
|---:|---|
| 0 | Node rất sạch |
| Cao | Node bị lẫn nhiều class |

Ví dụ node có 5 Pass và 5 Fail:

$$
p_{pass} = 0.5
$$

$$
p_{fail} = 0.5
$$

$$
Entropy = -[0.5\log_2(0.5) + 0.5\log_2(0.5)]
$$

$$
Entropy = 1
$$

Kết luận:

```text
Entropy = 1 → dữ liệu bị lẫn hoàn toàn.
```

In [ ]:
# Tính Entropy bằng Python
import math

def entropy(counts):
    total = sum(counts)
    result = 0
    for count in counts:
        if count == 0:
            continue
        p = count / total
        result -= p * math.log2(p)
    return result

# Ví dụ: 5 Pass, 5 Fail
entropy_value = entropy([5, 5])
entropy_value

## 5. Information Gain

**Information Gain** đo xem sau khi chia nhánh, cây giảm được bao nhiêu độ hỗn loạn.

Công thức:

$$
Information\ Gain = Entropy(parent) - Entropy(children)
$$

Trong đó:

$$
Entropy(children) = \frac{n_1}{n}Entropy(child_1) + \frac{n_2}{n}Entropy(child_2)
$$

Ý nghĩa:

```text
Information Gain cao → split tốt
Information Gain thấp → split kém
```

Decision Tree sẽ ưu tiên split có **Information Gain cao nhất** nếu dùng Entropy.

Nếu dùng Gini, cây thường chọn split có **Weighted Gini thấp nhất**.

## 6. Ví dụ tính tay: chọn split tốt hơn

Ban đầu có node cha:

| Positive | Negative |
|---:|---:|
| 6 | 6 |

Tổng cộng 12 mẫu.

Có 2 cách chia:

### Split A

| Nhánh | Positive | Negative |
|---|---:|---:|
| Left | 5 | 1 |
| Right | 1 | 5 |

### Split B

| Nhánh | Positive | Negative |
|---|---:|---:|
| Left | 3 | 3 |
| Right | 3 | 3 |

Câu hỏi:

```text
Split nào tốt hơn?
```

In [ ]:
# So sánh Split A và Split B bằng Gini

def weighted_gini(left_counts, right_counts):
    n_left = sum(left_counts)
    n_right = sum(right_counts)
    n_total = n_left + n_right
    return (n_left / n_total) * gini(left_counts) + (n_right / n_total) * gini(right_counts)

split_A_gini = weighted_gini([5, 1], [1, 5])
split_B_gini = weighted_gini([3, 3], [3, 3])

print("Weighted Gini Split A:", split_A_gini)
print("Weighted Gini Split B:", split_B_gini)

if split_A_gini < split_B_gini:
    print("Split A tốt hơn vì Weighted Gini thấp hơn.")
else:
    print("Split B tốt hơn vì Weighted Gini thấp hơn.")

Giải thích:

```text
Split A tạo ra 2 nhánh khá sạch:
Left: 5 Positive, 1 Negative
Right: 1 Positive, 5 Negative
```

Trong khi đó:

```text
Split B vẫn chia ra 2 nhóm bị lẫn hoàn toàn:
Left: 3 Positive, 3 Negative
Right: 3 Positive, 3 Negative
```

Vì vậy:

```text
Split A tốt hơn Split B.
```

## 7. Ví dụ threshold: Hours Study

Dữ liệu:

| Student | Hours Study | Result |
|---|---:|---|
| A | 1 | Fail |
| B | 2 | Fail |
| C | 3 | Fail |
| D | 6 | Pass |
| E | 7 | Pass |
| F | 8 | Pass |

Một threshold hợp lý:

```text
Hours Study <= 4?
```

Kết quả:

```text
Hours Study <= 4 → Fail, Fail, Fail
Hours Study > 4  → Pass, Pass, Pass
```

Hai nhánh đều pure hoàn toàn.

In [ ]:
import pandas as pd

study_data = pd.DataFrame({
    "Hours_Study": [1, 2, 3, 6, 7, 8],
    "Result": ["Fail", "Fail", "Fail", "Pass", "Pass", "Pass"]
})

study_data

In [ ]:
# Chia dữ liệu theo threshold Hours_Study <= 4
left_node = study_data[study_data["Hours_Study"] <= 4]
right_node = study_data[study_data["Hours_Study"] > 4]

print("Left node:")
print(left_node)

print("\nRight node:")
print(right_node)

## 8. Demo Python với sklearn

Ta sẽ demo Decision Tree bằng dataset có sẵn trong `sklearn`: **Iris dataset**.

Bài toán:

```text
Dự đoán loài hoa Iris dựa trên chiều dài/rộng của sepal và petal.
```

Đây là bài toán **classification**.

In [ ]:
# Import thư viện
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [ ]:
# Load Iris dataset
iris = load_iris()

X = pd.DataFrame(iris.data, columns=iris.feature_names)
y = pd.Series(iris.target, name="target")

# Xem 5 dòng đầu
X.head()

In [ ]:
# Tên các class
iris.target_names

In [ ]:
# Chia train/test
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Train size:", X_train.shape)
print("Test size:", X_test.shape)

In [ ]:
# Tạo model Decision Tree
model = DecisionTreeClassifier(
    criterion="gini",     # dùng Gini Impurity
    max_depth=3,          # giới hạn độ sâu để tránh overfitting
    random_state=42
)

# Train model
model.fit(X_train, y_train)

In [ ]:
# Dự đoán
y_pred = model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
print("Accuracy:", accuracy)

In [ ]:
# Báo cáo kết quả chi tiết
print(classification_report(y_test, y_pred, target_names=iris.target_names))

In [ ]:
# Confusion Matrix
confusion_matrix(y_test, y_pred)

### Vẽ cây quyết định

Hình dưới giúp ta nhìn được Decision Tree đã chia nhánh như thế nào.

In [ ]:
plt.figure(figsize=(16, 8))
plot_tree(
    model,
    feature_names=iris.feature_names,
    class_names=iris.target_names,
    filled=True,
    rounded=True
)
plt.show()

## 9. Giải thích demo

Trong cây trên, mỗi node thường có các thông tin:

| Thành phần | Ý nghĩa |
|---|---|
| Feature <= threshold | Điều kiện chia nhánh |
| gini | Độ lẫn lộn của node |
| samples | Số mẫu trong node |
| value | Số lượng mẫu thuộc từng class |
| class | Class dự đoán tại node đó |

Ví dụ một node có:

```text
gini = 0.0
```

Nghĩa là node đó rất sạch, tất cả mẫu trong node thuộc cùng một class.

Ví dụ:

```text
value = [0, 40, 0]
```

Nghĩa là node đó có:

```text
0 mẫu class 0
40 mẫu class 1
0 mẫu class 2
```

Vậy class dự đoán là class 1.

## 10. Overfitting trong Decision Tree

Decision Tree rất dễ bị overfitting nếu cây quá sâu.

Ví dụ:

```text
Cây quá sâu → học thuộc training data
Cây vừa phải → học được quy luật tổng quát
```

Một số tham số quan trọng để giảm overfitting:

| Tham số | Ý nghĩa |
|---|---|
| `max_depth` | Giới hạn độ sâu của cây |
| `min_samples_split` | Số mẫu tối thiểu để tiếp tục chia node |
| `min_samples_leaf` | Số mẫu tối thiểu ở leaf node |
| `ccp_alpha` | Dùng cho pruning |

In [ ]:
# So sánh cây nông và cây sâu
shallow_tree = DecisionTreeClassifier(max_depth=2, random_state=42)
deep_tree = DecisionTreeClassifier(max_depth=None, random_state=42)

shallow_tree.fit(X_train, y_train)
deep_tree.fit(X_train, y_train)

print("Shallow tree train accuracy:", shallow_tree.score(X_train, y_train))
print("Shallow tree test accuracy:", shallow_tree.score(X_test, y_test))

print("\nDeep tree train accuracy:", deep_tree.score(X_train, y_train))
print("Deep tree test accuracy:", deep_tree.score(X_test, y_test))

Giải thích:

```text
Nếu train accuracy rất cao nhưng test accuracy thấp hơn nhiều
→ có dấu hiệu overfitting.
```

Trong thực tế, ta không nên chỉ nhìn accuracy trên training data.

Phải luôn kiểm tra trên test data.

## 11. Ưu điểm và nhược điểm của Decision Tree

### Ưu điểm

| Ưu điểm | Giải thích |
|---|---|
| Dễ hiểu | Cấu trúc giống flowchart |
| Dễ giải thích | Có thể trình bày bằng rule IF-ELSE |
| Không cần scale dữ liệu | Không phụ thuộc vào khoảng cách |
| Dùng được cho classification và regression | Linh hoạt |
| Xử lý được quan hệ phi tuyến | Không bị bó buộc bởi đường thẳng |

### Nhược điểm

| Nhược điểm | Giải thích |
|---|---|
| Dễ overfitting | Cây quá sâu có thể học thuộc dữ liệu |
| Không ổn định | Dữ liệu thay đổi nhỏ có thể làm cây khác đi |
| Một cây đơn lẻ thường chưa mạnh | Random Forest/XGBoost thường tốt hơn |
| Có thể bias với feature nhiều giá trị | Một số feature có nhiều cách chia dễ được ưu tiên |

## 12. Câu trả lời mẫu: Cách cây quyết định chia nhánh và ưu nhược điểm

Decision Tree chia nhánh bằng cách thử nhiều feature và threshold khác nhau, sau đó chọn cách chia làm cho dữ liệu sau khi tách ra trở nên pure nhất. Độ pure thường được đo bằng Gini Impurity hoặc Entropy. Nếu dùng Gini, cây chọn split có Weighted Gini thấp nhất. Nếu dùng Entropy, cây chọn split có Information Gain cao nhất. Mỗi lần chia, cây cố gắng giảm sự lẫn lộn giữa các class trong từng node. Quá trình này tiếp tục cho đến khi node đủ sạch, đạt độ sâu tối đa, hoặc không còn đủ dữ liệu để chia tiếp.

Ưu điểm của Decision Tree là dễ hiểu, dễ giải thích, không cần scale dữ liệu, dùng được cho cả classification và regression, đồng thời xử lý được dữ liệu phi tuyến. Nhược điểm là dễ overfitting nếu cây quá sâu, không ổn định khi dữ liệu thay đổi nhỏ, và một cây đơn lẻ thường không mạnh bằng các mô hình ensemble như Random Forest hoặc XGBoost.

## 13. Bài tập luyện tập

### Bài 1: Tính Gini

Một node có:

| Class | Count |
|---|---:|
| Positive | 9 |
| Negative | 1 |

Yêu cầu:

```text
Tính Gini Impurity của node này.
```

---

### Bài 2: Tính Entropy

Một node có:

| Class | Count |
|---|---:|
| Pass | 4 |
| Fail | 4 |

Yêu cầu:

```text
Tính Entropy của node này.
```

---

### Bài 3: Chọn split tốt hơn

Split A:

| Nhánh | Positive | Negative |
|---|---:|---:|
| Left | 4 | 0 |
| Right | 1 | 5 |

Split B:

| Nhánh | Positive | Negative |
|---|---:|---:|
| Left | 3 | 2 |
| Right | 2 | 3 |

Yêu cầu:

```text
Dùng Weighted Gini để xác định split nào tốt hơn.
```

---

### Bài 4: Giải thích overfitting

Có 2 cây:

| Model | Train Accuracy | Test Accuracy |
|---|---:|---:|
| Tree A | 92% | 89% |
| Tree B | 100% | 65% |

Yêu cầu:

```text
Cây nào bị overfitting? Vì sao?
```

## 14. Đáp án bài tập

### Bài 1

$$
p_{pos} = \frac{9}{10} = 0.9
$$

$$
p_{neg} = \frac{1}{10} = 0.1
$$

$$
Gini = 1 - (0.9^2 + 0.1^2)
$$

$$
Gini = 1 - (0.81 + 0.01) = 0.18
$$

---

### Bài 2

$$
p_{pass} = 0.5
$$

$$
p_{fail} = 0.5
$$

$$
Entropy = -[0.5\log_2(0.5) + 0.5\log_2(0.5)] = 1
$$

---

### Bài 3

Split A tốt hơn vì tạo ra một nhánh hoàn toàn pure:

```text
Left: 4 Positive, 0 Negative
```

Trong khi Split B vẫn còn lẫn khá nhiều ở cả hai nhánh.

Có thể kiểm tra bằng code ở cell dưới.

---

### Bài 4

Tree B bị overfitting.

Lý do:

```text
Train accuracy = 100%
Test accuracy = 65%
```

Cây học rất tốt trên training data nhưng dự đoán kém trên dữ liệu mới.

In [ ]:
# Kiểm tra đáp án bài 3 bằng Python
split_A = weighted_gini([4, 0], [1, 5])
split_B = weighted_gini([3, 2], [2, 3])

print("Weighted Gini Split A:", split_A)
print("Weighted Gini Split B:", split_B)

if split_A < split_B:
    print("Split A tốt hơn.")
else:
    print("Split B tốt hơn.")